# Shrink a bloated system prompt with Alexandria

You maintain the system prompt for **TripGenie**, a travel-planning chatbot. Over months, every time
something went wrong you bolted on "just one more line." Now each section is a pile of instructions
that say almost the same thing three different ways — the prompt is long, repetitive, and you pay for
every one of those tokens on every single call.

**[Alexandria](https://github.com/ucsc-cse115a-alexandria/alexandria)** shrinks instruction-heavy
prompts *without labels, training, or a target answer*. It finds near-duplicate sentences with
embeddings and has an LLM merge each pair into one tighter sentence that keeps the meaning. Markdown
headings are protected, so it compacts your prompt **section by section**.

This notebook runs the whole loop with the `alexandria` **CLI**:

1. Look at the bloated prompt.
2. `alexandria reduce` it — one command.
3. See exactly what merged (the **diff**).
4. **Score** it — tokens saved vs. meaning kept.
5. Prove the shorter prompt still drives the **same chatbot behavior**.

> **Requirements:** Python 3.14, [uv](https://docs.astral.sh/uv/), and an OpenAI key. This notebook
> makes real OpenAI calls (embeddings for the reduction + a few `gpt-4o-mini` chat completions);
> total cost is a fraction of a cent. Provide the key via a `.env` file (`OPENAI_API_KEY=...`) or an
> exported environment variable.

## Setup

In [1]:
import json
import subprocess
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()          # picks up a .env in this folder or any parent
client = OpenAI()      # raises immediately if no OPENAI_API_KEY is resolvable

CHAT_MODEL = "gpt-4o-mini"          # any chat model your key can reach — swap freely
PROMPT_PATH = Path("travel_agent_prompt.md")


def alexandria(*args: str) -> str:
    # Run the alexandria CLI and return its stdout (raises on a non-zero exit).
    done = subprocess.run(
        ["uv", "run", "alexandria", *args],
        capture_output=True, text=True, check=True,
    )
    return done.stdout


print("alexandria CLI:", alexandria("--help").splitlines()[0])

alexandria CLI: Usage: alexandria [OPTIONS] COMMAND [ARGS]...


## Step 1 — The prompt you're maintaining

Here's the current `TripGenie` system prompt. Four sections, and each one has grown to **six bullets**
that overlap heavily — notice how every pair of lines is really saying the same thing in slightly
different words. This is exactly the kind of drift that accumulates in a real production prompt.

In [2]:
prompt_text = PROMPT_PATH.read_text()
print(prompt_text)

# Role

- You are TripGenie, a travel-planning assistant that helps users plan their trips.
- You are TripGenie, an AI travel companion that helps people organize their vacations.
- You build detailed day-by-day itineraries for the whole trip.
- You put together a full day-by-day itinerary covering the entire journey.
- You provide practical guidance on booking flights, hotels, and transportation.
- You offer practical advice on arranging flights, hotels, and transportation.

# Communication

- Always be warm, friendly, and polite when you greet and reply to every traveler.
- Always be welcoming, friendly, and courteous in every message you send to a traveler.
- Be patient and empathetic with travelers who feel stressed or anxious about their trips.
- Be understanding and reassuring with travelers who feel worried or anxious about their travel plans.
- Always keep a professional and respectful tone, and never be rude or dismissive to a traveler.
- Always maintain a courteous and respec

## Step 2 — Reduce it with one command

```bash
alexandria reduce travel_agent_prompt.md          # reduced prompt on stdout
alexandria reduce travel_agent_prompt.md --json   # machine-readable summary
```

We use `--json` so we can inspect *what* changed. The LLM rewrites are stochastic, so exact wording
varies run to run — but which near-duplicate pairs get merged, and the overall reduction, are stable.

In [3]:
result = json.loads(alexandria("reduce", str(PROMPT_PATH), "--json"))
reduced_text = result["text"]

print(f"{result['source_tokens']} -> {result['reduced_tokens']} tokens "
      f"({result['reduction_pct'] * 100:.0f}% smaller)\n")
print(reduced_text)

450 -> 241 tokens (46% smaller)

# Role

You are TripGenie, an AI travel-planning assistant and companion that helps people organize their trips and vacations.
Build a detailed, complete day-by-day itinerary covering the entire trip.
Provide practical guidance and advice on booking and arranging flights, hotels, and transportation.

# Communication

Always be warm, welcoming, friendly, and courteous in every message to travelers.
Be patient, empathetic, understanding, and reassuring toward travelers who feel stressed or anxious about their trips or travel plans.
Always maintain a professional, courteous, and respectful tone; never be rude, dismissive, or condescending to a traveler.

# Accuracy

Do not invent or claim prices, schedules, or availability you have not been given or cannot confirm.
Tell the user when you are uncertain about an answer rather than guessing.
Advise travelers to verify prices and times on the official airline or hotel website before booking.

# Output Format



## Step 3 — See what merged (the diff)

Each section keeps its heading and collapses from **6 near-duplicate bullets to ~3 tight sentences**.
Alexandria reports, for every merge, the cosine similarity of the pair it fused — higher means the two
lines were more nearly identical in meaning.

In [4]:
def sections(md_text: str) -> dict[str, list[str]]:
    # Split a prompt into {heading: [non-empty lines]} keyed by its Markdown headings.
    out: dict[str, list[str]] = {}
    name = None
    for line in md_text.splitlines():
        if line.startswith("# "):
            name = line[2:].strip()
            out[name] = []
        elif line.strip() and name:
            out[name].append(line.strip().lstrip("- "))
    return out


before, after = sections(prompt_text), sections(reduced_text)
for name in before:
    print(f"# {name}   ({len(before[name])} lines -> {len(after[name])} lines)")
    for line in before[name]:
        print("   -", line)
    print("   " + "-" * 3)
    for line in after[name]:
        print("   =>", line)
    print()

print("Merges applied (cosine = how near-duplicate the fused pair was):")
for edit in result["applied"]:
    print("  *", edit["reason"])

# Role   (6 lines -> 3 lines)
   - You are TripGenie, a travel-planning assistant that helps users plan their trips.
   - You are TripGenie, an AI travel companion that helps people organize their vacations.
   - You build detailed day-by-day itineraries for the whole trip.
   - You put together a full day-by-day itinerary covering the entire journey.
   - You provide practical guidance on booking flights, hotels, and transportation.
   - You offer practical advice on arranging flights, hotels, and transportation.
   ---
   => You are TripGenie, an AI travel-planning assistant and companion that helps people organize their trips and vacations.
   => Build a detailed, complete day-by-day itinerary covering the entire trip.
   => Provide practical guidance and advice on booking and arranging flights, hotels, and transportation.

# Communication   (6 lines -> 3 lines)
   - Always be warm, friendly, and polite when you greet and reply to every traveler.
   - Always be welcoming, friendly, 

## Step 4 — Score it: tokens saved vs. meaning kept

`alexandria compare` embeds both prompts and reports their cosine similarity alongside the token
reduction. `--min-similarity` turns that into an exit-code gate you can drop into CI to guarantee a
reduction never drifts too far from the original meaning.

In [5]:
reduced_path = Path(tempfile.mkdtemp()) / "reduced.md"
reduced_path.write_text(reduced_text)

score = json.loads(alexandria("compare", str(PROMPT_PATH), str(reduced_path)))
print(f"tokens:      {score['original_tokens']} -> {score['edited_tokens']} "
      f"({score['token_reduction'] * 100:.0f}% fewer)")
print(f"similarity:  {score['similarity']:.3f}   (1.0 = identical meaning)")
print(f"drift:       {score['cos_sim_diff']:.3f}   (1 - similarity)")

gate = subprocess.run(
    ["uv", "run", "alexandria", "compare", str(PROMPT_PATH), str(reduced_path),
     "--min-similarity", "0.9"],
    capture_output=True, text=True,
)
print(f"\npasses `--min-similarity 0.9` gate: {gate.returncode == 0}")

tokens:      450 -> 241 (46% fewer)
similarity:  0.932   (1.0 = identical meaning)
drift:       0.068   (1 - similarity)



passes `--min-similarity 0.9` gate: True


## Step 5 — Does the shorter prompt still work?

Token counts are only half the story. The real question: does the compressed prompt still make the
chatbot **behave the same**? We wire both prompts into the same `gpt-4o-mini` travel bot and ask three
questions, each probing a different section:

- **Plan a trip** -> exercises *Role* + *Output Format* (day-by-day itinerary, prices with currency).
- **Ask for a live price** -> exercises *Accuracy* (refuse to invent it, defer to official sources).
- **Vent about a cancelled flight** -> exercises *Communication* (empathetic, practical).

In [6]:
def ask(system_prompt: str, question: str) -> str:
    reply = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return reply.choices[0].message.content.strip()


questions = [
    "I have 3 days in Kyoto in April. Can you plan my trip?",
    "How much is a flight from San Francisco to Tokyo next month?",
    "I'm really stressed, my flight got cancelled. What do I do?",
]

runs = []
for question in questions:
    runs.append({
        "q": question,
        "long": ask(prompt_text, question),
        "short": ask(reduced_text, question),
    })
    print("=" * 78)
    print("Q:", question)
    print("\n--- long prompt ---\n", runs[-1]["long"][:650])
    print("\n--- short prompt ---\n", runs[-1]["short"][:650])
    print()

Q: I have 3 days in Kyoto in April. Can you plan my trip?

--- long prompt ---
 Absolutely! Here’s a day-by-day itinerary for your 3-day trip to Kyoto in April:

### Day 1: Explore Eastern Kyoto
- **Morning:**
  - Visit **Kiyomizu-dera Temple** (Entrance: ¥400)
  - Stroll through **Sannenzaka and Ninenzaka** streets for traditional shops and tea houses.
  
- **Lunch:**
  - Enjoy a meal at a local restaurant in the area (approx. ¥1,200).

- **Afternoon:**
  - Head to **Ginkaku-ji (Silver Pavilion)** (Entrance: ¥500).
  - Walk along the **Philosopher's Path**, lined with cherry blossoms.

- **Evening:**
  - Dinner in the **Gion district** (approx. ¥2,000).
  - Optional: Attend a traditional **Geisha performance** (prices v

--- short prompt ---
 Absolutely! Kyoto is a beautiful city with rich history and stunning scenery. Here’s a detailed 3-day itinerary for your trip in April:

### Day 1: Explore Eastern Kyoto

**Morning:**
- **Kiyomizu-dera Temple**  
  - Visit this iconic temple know

Q: How much is a flight from San Francisco to Tokyo next month?

--- long prompt ---
 I can't provide specific prices for flights, as they can vary widely based on the airline, time of booking, and availability. I recommend checking official airline websites or flight comparison platforms for the most accurate and up-to-date pricing for flights from San Francisco to Tokyo next month. If you need help with anything else, feel free to ask!

--- short prompt ---
 I don't have real-time access to flight prices or availability. I recommend checking popular travel websites or airline websites directly for the most accurate and up-to-date information on flights from San Francisco to Tokyo next month. You can also consider using flight comparison tools to find the best deals. If you need help with anything else, feel free to ask!



Q: I'm really stressed, my flight got cancelled. What do I do?

--- long prompt ---
 I'm really sorry to hear that your flight got canceled. Here are some steps you can take to help manage the situation:

1. **Contact the Airline**: Reach out to the airline as soon as possible. They can provide information on rebooking options or refunds.

2. **Check for Alternative Flights**: Look for other flights to your destination. You can use the airline's website or travel booking platforms.

3. **Know Your Rights**: Familiarize yourself with the airline's policies regarding cancellations. Depending on the reason for the cancellation, you may be entitled to compensation or accommodations.

4. **Stay Calm**: Take a deep breath. It’s und

--- short prompt ---
 I'm really sorry to hear that your flight got canceled. It can be very stressful, but let's take it step by step to help you through this.

1. **Check for Notifications**: Look for any emails or messages from the airline regarding your fligh

### Judge it automatically

Eyeballing three answers is fine; let's also score it. We ask `gpt-4o-mini` to judge **behavioral**
equivalence — same instructions followed — while explicitly ignoring length, formatting, and wording.

In [7]:
JUDGE = (
    "You compare two travel-assistant answers to the same question. One assistant used a long "
    "system prompt; the other used a compressed version of that same prompt. Judge BEHAVIORAL "
    "equivalence only -- whether both follow the same instructions: give a day-by-day itinerary "
    "when asked to plan a trip, label prices with a currency, refuse to invent specific "
    "prices/times and defer to official sources, and stay empathetic and practical. IGNORE "
    "differences in length, formatting, and exact wording. Return JSON: "
    '{"equivalent": true/false, "reason": "<one sentence about the shared behavior>"}.'
)


def behaves_the_same(question: str, long_answer: str, short_answer: str) -> dict:
    body = (f"USER: {question}\n\nASSISTANT A (long prompt):\n{long_answer}"
            f"\n\nASSISTANT B (short prompt):\n{short_answer}")
    reply = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": JUDGE},
                  {"role": "user", "content": body}],
    )
    return json.loads(reply.choices[0].message.content)


equivalent = 0
for r in runs:
    verdict = behaves_the_same(r["q"], r["long"], r["short"])
    equivalent += verdict["equivalent"]
    print("[ok]" if verdict["equivalent"] else "[x]", r["q"])
    print("    ", verdict["reason"])

print(f"\nBehaviorally equivalent on {equivalent}/{len(runs)} questions.")

[ok] I have 3 days in Kyoto in April. Can you plan my trip?
     Both assistants provided a day-by-day itinerary for Kyoto, included currency labels for prices, refrained from inventing specific prices/times, and maintained an empathetic and practical tone.


[ok] How much is a flight from San Francisco to Tokyo next month?
     Both assistants refuse to provide specific prices and recommend checking official sources for accurate information.


[ok] I'm really stressed, my flight got cancelled. What do I do?
     Both assistants provide empathetic, practical steps for managing a flight cancellation, including contacting the airline, exploring rebooking options, and considering accommodation, while deferring to official sources for specific prices and times.

Behaviorally equivalent on 3/3 questions.


## Takeaways

- **~46% fewer tokens** on every call, at **~0.93 cosine similarity** to the original meaning.
- The compressed prompt still produces a day-by-day itinerary, still refuses to invent prices, and
  still answers a stressed traveler with empathy — **behaviorally equivalent** on all three probes.
- The whole loop is one CLI command (`alexandria reduce`) plus a similarity gate you can put in CI.

Try it on your own prompt:

```bash
alexandria reduce your_prompt.md > reduced.md
alexandria compare your_prompt.md reduced.md --min-similarity 0.9
```

Prefer to approve edits yourself? Add `--interactive` (terminal) or `--browser` to review each
proposed merge and apply only the ones you accept.